# Ingesta Inicial del Dataset con Validacion de Esquema

Este bloque se divide en tres pasos:

1. Inicializar la sesion de Spark.
2. Definir el esquema explicito con `StructType`.
3. Cargar el CSV, validar que el esquema coincide y revisar una muestra inicial.

In [1]:
import os
import socket
from urllib.parse import urlparse

from pyspark.sql import SparkSession

# Limpiar cualquier sesión anterior detenida
active_session = SparkSession.getActiveSession()
if active_session is not None:
    try:
        active_session.stop()
    except:
        pass

master_url = os.getenv("SPARK_MASTER", "spark://spark-master:7077")
use_remote_master = os.getenv("SPARK_USE_REMOTE_MASTER", "").strip().lower() in {"1", "true", "yes"}


def master_is_reachable(master: str) -> bool:
    parsed = urlparse(master)
    if parsed.scheme != "spark" or not parsed.hostname or not parsed.port:
        return False
    try:
        with socket.create_connection((parsed.hostname, parsed.port), timeout=2):
            return True
    except OSError:
        return False


spark_master = master_url if use_remote_master and master_is_reachable(master_url) else "local[*]"
if spark_master == "local[*]":
    print("Using local[*] so the smoke test does not depend on remote worker Python versions.")
else:
    print(f"Using remote Spark master: {spark_master}")

spark = (SparkSession.builder
         .appName("IABurnout-Spark-Smoke-Test")
         .master(spark_master)
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

Using local[*] so the smoke test does not depend on remote worker Python versions.
Spark version: 4.1.1
Spark master: local[*]


In [2]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

schema = StructType([
    StructField("record_id", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("job_role", StringType(), True),
    StructField("employment_type", StringType(), True),
    StructField("work_model", StringType(), True),
    StructField("company_size", StringType(), True),
    StructField("age_group", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("years_of_experience", IntegerType(), True),
    StructField("annual_salary_usd", IntegerType(), True),
    StructField("weekly_work_hours", IntegerType(), True),
    StructField("weekly_overtime_hours", IntegerType(), True),
    StructField("mental_health_condition", StringType(), True),
    StructField("has_diagnosis", StringType(), True),
    StructField("treatment_type", StringType(), True),
    StructField("stress_level", StringType(), True),
    StructField("burnout_risk_score", DoubleType(), True),
    StructField("work_life_balance_score", DoubleType(), True),
    StructField("productivity_score", DoubleType(), True),
    StructField("job_satisfaction_score", DoubleType(), True),
    StructField("absenteeism_days_per_year", IntegerType(), True),
    StructField("employer_support_level", StringType(), True),
    StructField("mental_health_policy_exists", StringType(), True),
    StructField("eap_available", StringType(), True),
    StructField("used_eap", StringType(), True),
    StructField("workplace_stigma_felt", StringType(), True),
    StructField("manager_support_score", DoubleType(), True),
    StructField("team_collaboration_score", DoubleType(), True),
    StructField("intention_to_leave", StringType(), True),
    StructField("remote_work_preference", StringType(), True),
    StructField("exercise_days_per_week", IntegerType(), True),
    StructField("sleep_hours_per_night", DoubleType(), True),
])

print(f"Esquema definido con {len(schema.fields)} columnas")

Esquema definido con 34 columnas


In [3]:
from pathlib import Path
import os

# Función para buscar recursivamente hacia arriba hasta encontrar la carpeta 'data'
def find_data_file(filename="mental_health_workplace.csv", max_depth=5):
    """Busca el archivo CSV subiendo en el árbol de directorios."""
    current = Path.cwd()
    for _ in range(max_depth):
        candidate = current / "data" / filename
        if candidate.exists():
            return candidate
        current = current.parent
    return None

# Intenta 1: Búsqueda recursiva desde el directorio actual
dataset_path = find_data_file()

# Intenta 2: Si está en un contenedor Docker, busca en rutas comunes de montaje
if dataset_path is None:
    for docker_mount in ["/work", "/workspace", "/app", "/home/jovyan/work"]:
        candidate = Path(docker_mount) / "data" / "mental_health_workplace.csv"
        if candidate.exists():
            dataset_path = candidate
            break

# Intenta 3: Si SPARK_HOME está configurado, búsqueda desde ahí
if dataset_path is None:
    spark_home = os.getenv("SPARK_HOME")
    if spark_home:
        current = Path(spark_home)
        for _ in range(5):
            candidate = current / "data" / "mental_health_workplace.csv"
            if candidate.exists():
                dataset_path = candidate
                break
            current = current.parent

if dataset_path is None:
    # Debug: mostrar donde estamos buscando
    print(f"❌ No se encontró el archivo!")
    print(f"Directorio de trabajo actual: {Path.cwd()}")
    print(f"SPARK_HOME: {os.getenv('SPARK_HOME', 'No configurado')}")
    print(f"\nIntentando listar directorios:")
    for d in [Path.cwd(), Path.cwd().parent, Path("/work"), Path("/workspace")]:
        if d.exists():
            print(f"\n  {d}:")
            try:
                items = sorted(d.iterdir())[:10]
                for item in items:
                    print(f"    - {item.name}{'/' if item.is_dir() else ''}")
            except PermissionError:
                print(f"    (sin permisos)")
    raise FileNotFoundError(
        "No se encontro mental_health_workplace.csv. "
        "Verifica que el archivo esté en data/ dentro del proyecto."
    )

dataset_path = str(dataset_path)
print(f"✓ Archivo encontrado: {dataset_path}")

df_burnout = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(schema)
    .csv(dataset_path)
)

expected_schema = [(f.name, f.dataType.simpleString()) for f in schema.fields]
actual_schema = [(f.name, f.dataType.simpleString()) for f in df_burnout.schema.fields]

if expected_schema != actual_schema:
    actual_schema_dict = dict(actual_schema)
    expected_schema_dict = dict(expected_schema)
    missing = [c for c, _ in expected_schema if c not in actual_schema_dict]
    unexpected = [c for c, _ in actual_schema if c not in expected_schema_dict]
    type_mismatch = [
        (name, exp_t, actual_schema_dict.get(name))
        for name, exp_t in expected_schema
        if name in actual_schema_dict and actual_schema_dict[name] != exp_t
    ]
    raise ValueError(
        "Esquema invalido tras ingesta. "
        f"Faltantes: {missing}. "
        f"Sobrantes: {unexpected}. "
        f"Tipos distintos: {type_mismatch}"
    )

print(f"Dataset cargado correctamente desde: {dataset_path}")
print(f"Filas: {df_burnout.count()} | Columnas: {len(df_burnout.columns)}")
df_burnout.printSchema()
df_burnout.show(5, truncate=False)

✓ Archivo encontrado: /data/mental_health_workplace.csv
Dataset cargado correctamente desde: /data/mental_health_workplace.csv
Filas: 10000 | Columnas: 34
root
 |-- record_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- work_model: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- years_of_experience: integer (nullable = true)
 |-- annual_salary_usd: integer (nullable = true)
 |-- weekly_work_hours: integer (nullable = true)
 |-- weekly_overtime_hours: integer (nullable = true)
 |-- mental_health_condition: string (nullable = true)
 |-- has_diagnosis: string (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- stress_level: string (nullable = true)
 |-- burnout_risk_score: double (nu